In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob, scipy
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from optax.losses import huber_loss
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, CNN, HybridNet, AttentionGNN
from jaxpm import camels, plotting, hpm, nn, graph, data, diagnostics

# print(jax.devices("gpu"))
print(jax.default_backend())

gpu


# configuration

In [3]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
# mesh_per_dim = 2 * parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# i_snapshots = np.arange(0, 2, dtype=int)
# i_snapshots = np.arange(-2, 0, dtype=int)
# i_snapshots = np.arange(0, 4, dtype=int)
# i_snapshots = np.arange(0, 8, dtype=int)
# i_snapshots = np.arange(-8, 0, dtype=int)
i_snapshots = np.arange(-4, 0, dtype=int)
# i_snapshots = np.arange(-16, 0, dtype=int)
# i_snapshots = range(1, 33+8, 8)
# i_snapshots = range(1, 33+4, 4)
# i_snapshots = None

# CAMELS

In [4]:
train_dict = camels.load_CV_snapshots(
    "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
    return_hydro=True,
)

cosmo = train_dict["cosmo"]
scales = train_dict["scales"]

dm_poss = train_dict["dm_poss"]
dm_vels = train_dict["dm_vels"]

gas_poss = train_dict["gas_poss"]
gas_vels = train_dict["gas_vels"]

Loaded /cluster/work/refregier/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=64,mesh=64,i=[-4 -3 -2 -1].h5


In [5]:
(scales[-1] - scales[0])/50

0.0026720204513207137

In [6]:
# vali_dict = camels.load_CV_snapshots(
#     "CV_1",
#     mesh_per_dim,
#     parts_per_dim,
#     i_snapshots=i_snapshots,
#     return_hydro=True,
# )

In [7]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

In [8]:
@nnx.jit(static_argnames=("loss_fn",))
def train_step(model, optimizer, loss_fn):
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    squared_sum = jax.tree_util.tree_reduce(
            lambda x, y: x + jnp.sum(y**2),
            grads,
            0.0
        )
    grad_norm = jnp.sqrt(squared_sum)
    return loss, grad_norm

losses = []

In [9]:
# @nnx.jit(static_argnames=("loss_fn",))
# def train_step(model, optimizer, loss_fn):
#     loss, grads = nnx.value_and_grad(loss_fn)(model)
#     optimizer.update(grads)

#     squared_sum = jax.tree_util.tree_reduce(
#             lambda x, y: x + jnp.sum(y**2),
#             grads,
#             0.0
#         )
#     grad_norm = jnp.sqrt(squared_sum)
#     return loss, grad_norm

# losses = []

In [10]:
def solve_ode_diffrax(model, architecture, training=False):    
    res = diffeqsolve(
            # terms=ODETerm(hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, gravity_model=None, pressure_model=model, gas_architecture=architecture, training=training)),
            terms=ODETerm(hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, gravity_model=None, pressure_model=model, gas_architecture=architecture)),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            # dt0=0.001,
            dt0=(scales[-1] - scales[0])/(2*len(scales)),
            # dt0=0.01,
            # dt0=0.04,
            # dt0=0.005,
            y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
            saveat=SaveAt(ts=scales),
            # max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res

In [11]:
(scales[-1] - scales[0])/(2*len(scales))

0.01670012782075446

# loss

### CAMELS ground truth

In [12]:
# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# field-level reference
gas_mass = cosmo.Omega_b / (cosmo.Omega_b + cosmo.Omega_c)
ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

# power spectrum reference
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
_, ref_cls = vpower_spectrum(ref_rho)

vcross_correlation_separate = jax.vmap(
    lambda field_a, field_b:
        cross_correlation_coefficients(
            compensate_cic(field_a),
            compensate_cic(field_b),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
vcross_correlation = lambda rhos: vcross_correlation_separate(rhos, ref_rho)

### particle-level

In [13]:
def particle_loss_fn(model, architecture, huber=False, pos_dead_zone=False):
    res = solve_ode_diffrax(model, architecture, training=True)
    gas_poss = res[2]
    gas_vels = res[3]

    delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
    pos_loss = jnp.sum(huber_loss(delta_pos), axis=-1)
    
    pos_loss = jnp.mean(pos_loss)
    
    res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
    kbins, res_cls = vpower_spectrum(res_rho)

    k = kbins[0]
    k_min, k_cutoff = k[0], k[int(0.2*len(k))]
    k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)

    cl_loss = jnp.sum(((res_cls/ref_cls - 1)*k_weights)**2, axis=-1)
    cl_loss = jnp.mean(cl_loss)

    return pos_loss + 0.1 * cl_loss

### field-level

In [14]:
# def field_loss_fn(model, architecture):
#     print("using voxel log MSE")
#     res = solve_ode_diffrax(model, architecture)
#     gas_poss = res[2]%mesh_per_dim
    
#     rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
    
#     eps = 1e-8
#     log_rho = jnp.log10(rho + eps)
#     log_ref_rho = jnp.log10(ref_rho + eps)
#     log_loss = jnp.mean((log_rho - log_ref_rho)**2)
        
#     return log_loss

In [15]:
def field_loss_fn(model, architecture, eps=1e-8):
    print("using voxel MSE")
    
    res = solve_ode_diffrax(model, architecture, training=True)
    gas_poss = res[2]%mesh_per_dim
    
    rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

    loss = (rho - ref_rho)**2
    # print(loss.shape)

    # loss = loss[-1]
    
    # loss /= jnp.maximum(scales.reshape(-1,1,1,1)**2, eps)
    loss = jnp.mean(loss)
        
    return loss

In [16]:
# loss = field_loss_fn(model, architecture)

In [17]:
# field_loss_fn(model, architecture)

In [18]:
# 1/scales**4

In [19]:
# def field_loss_fn(model, architecture):
#     print("using voxel relative error")

#     res = solve_ode_diffrax(model, architecture)
#     gas_poss = res[2]%mesh_per_dim
    
#     rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
    
#     eps = 1e-8
#     # relative error
#     rel_error = jnp.abs((rho - ref_rho) / (ref_rho + eps))

#     # # apply weighting to emphasize regions
#     # importance = jnp.sqrt(ref_rho + eps)  # Square root emphasizes medium densities
#     # weighted_error = rel_error * importance
    
#     # rel_loss = jnp.mean(jnp.sum(weighted_error, axis=(1,2,3)))

#     rel_loss = jnp.mean(rel_error)
    
#     return rel_loss

# architecture

### CNN

In [20]:
# model = CNN(
#     d_in=5, 
#     d_out=1,
#     d_hidden=64,
#     # n_hidden=8,
#     n_hidden=4,
#     # kernel_size=(5, 5, 5),
#     kernel_size=(3, 3, 3),
#     rngs=nnx.Rngs(0),
#     # use_residual=False,
#     use_residual=True,
#     # norm_type=False,
#     # norm_type="batch",
#     norm_type="layer",
#     # norm_type=None,
#     activation=jax.nn.swish,
# )

# architecture = "cnn"

In [28]:
from jaxpm.nn import CNN2

model = CNN2(
    d_in=5, 
    d_out=1,
    d_hidden=64,
    # n_hidden=8,
    n_hidden=4,
    # kernel_size=(5, 5, 5),
    kernel_size=(3, 3, 3),
    rngs=nnx.Rngs(0),
    # use_residual=False,
    use_residual=True,
    # norm_type=False,
    # norm_type="batch",
    norm_type="layer",
    # norm_type=None,
    activation=jax.nn.swish,
    dropout_rate=0,
)

architecture = "cnn"

In [29]:
# nn_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, gas_model=model, gas_architecture=architecture)

# nn_res = diffeqsolve(
#         terms=ODETerm(nn_ode),
#         solver=LeapfrogMidpoint(),
#         t0=scales[0],
#         t1=scales[-1],
#         dt0=0.01,
#         y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
#         saveat=SaveAt(ts=scales),
#         max_steps=100,
#         stepsize_controller=ConstantStepSize(),
# )

# nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels = nn_res.ys

In [30]:
# gas_inputs, field_inputs, _, _ = data.get_offline_regression_data(
#     train_dict,
#     x_labels=["rho", "fscalar", "vel_disp", "vel_div"],
#     y_labels=["P", "U", "T"],
# )

# print(gas_inputs.shape)
# print(field_inputs.shape)

# normalized_fields = nnx.LayerNorm(field_inputs.shape[-1], rngs=nnx.Rngs(0), reduction_axes=-1)(field_inputs)
# temp = (field_inputs - jnp.mean(field_inputs, axis=-1, keepdims=True))/jnp.std(field_inputs, axis=-1, keepdims=True)

In [31]:
# i = -1

# dm_force, gas_force = hpm.hpm_forces_cnn(
#     mesh_per_dim,
#     cosmo,
#     scales[i],
#     dm_poss[i],
#     gas_poss[i],
#     # gravity
#     dm_model=None,
#     # pressure
#     gas_model=model,
#     gas_vel=gas_vels[i],
#     gas_latent=None,
#     gas_architecture=architecture,
# )

### MLP + CNN

In [32]:
# mlp = MLP(
#     d_in=5,
#     d_out=8, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

# cnn = CNN(
#     d_in=4, 
#     d_out=8,
#     d_hidden=16,
#     n_hidden=1,
#     kernel_size=(3, 3, 3),
#     strides=1,
#     rngs=nnx.Rngs(0)
# )

# model = HybridNet(
#     mlp,
#     cnn,
#     d_out=1, 
#     rngs=nnx.Rngs(0)
# )

# architecture = "mlp+cnn"

# training

In [33]:
total_steps = 100
# learning_rate = 1e-4
# learning_rate = 1e-5
learning_rate = 1e-6
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-4, 
#     decay_steps=total_steps, 
#     alpha=0.01,
# )
clip_norm = 1.0

# optimizer = nnx.Optimizer(
#     model,
#     optax.chain(
#         optax.clip_by_global_norm(clip_norm),
#         optax.adam(learning_rate)
#         # optax.sgd(learning_rate)
#     )
# )

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        # optax.adaptive_grad_clip(clipping=1e-3, eps=1e-3),
        optax.clip_by_global_norm(clip_norm),
        # optax.ema(decay=0.999),
        # optax.adamw(learning_rate, b1=0.95, b2= 0.9999, eps=1e-5)
        optax.adam(learning_rate)
    )
)

losses = []
grad_norms = []
# loss_fn = lambda model: particle_loss_fn(model, architecture)
loss_fn = lambda model: field_loss_fn(model, architecture)

In [34]:
for i in (pbar := tqdm.tqdm(range(total_steps))):  
    loss, grad_norm = train_step(model, optimizer, loss_fn)
    
    losses.append(float(loss))
    grad_norms.append(float(grad_norm))
    pbar.set_description(f"Loss: {loss:.4f}, Grad norm: {grad_norm:.4f}")

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log", title="loss")

fig, ax = plt.subplots()
ax.plot(grad_norms)
ax.set(yscale="log", title="norm(grad)")

  0%|          | 0/100 [00:00<?, ?it/s]

using voxel MSE
dark matter and gas
Using CNN forces
Using learned pressure force


  0%|          | 0/100 [00:00<?, ?it/s]


ValueError: `terms` must be a PyTree of `AbstractTerms` (such as `ODETerm`), with structure <class 'diffrax._term.AbstractTerm'>

# run the simulation

In [ ]:
diagnostics.run_simulations(train_dict, mesh_per_dim, pressure_model=model, gas_architecture=architecture)

In [ ]:
# def run_simulations(camels_dict):
#     scales = camels_dict["scales"]
    
#     dm_poss = camels_dict["dm_poss"]
#     dm_vels = camels_dict["dm_vels"]
    
#     gas_poss = camels_dict["gas_poss"]
#     gas_vels = camels_dict["gas_vels"]

    
#     og_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo)
#     og_res = diffeqsolve(
#             terms=ODETerm(og_ode),
#             solver=LeapfrogMidpoint(),
#             t0=scales[0],
#             t1=scales[-1],
#             dt0=0.01,
#             y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
#             saveat=SaveAt(ts=scales),
#             max_steps=100,
#             stepsize_controller=ConstantStepSize(),
#     )
#     og_dm_poss, og_dm_vels, og_gas_poss, og_gas_vels = og_res.ys

#     nn_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, pressure_model=model, gas_architecture=architecture)
#     nn_res = diffeqsolve(
#             terms=ODETerm(nn_ode),
#             solver=LeapfrogMidpoint(),
#             t0=scales[0],
#             t1=scales[-1],
#             dt0=0.01,
#             y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
#             saveat=SaveAt(ts=scales),
#             max_steps=100,
#             stepsize_controller=ConstantStepSize(),
#     )
#     nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels = nn_res.ys

#     with jax.default_device(jax.devices("cpu")[0]):
#         plotting.compare_particle_evolution(
#             mesh_shape, 
#             scales, 
#             jnp.stack([gas_poss, og_gas_poss, nn_gas_poss], axis=0), 
#             title="gas",
#             col_titles=["CAMELS", "gravity", "gravity + pressure"],
#             include_pk=True,
#             include_reference=True,
#         )

In [ ]:
run_simulations(train_dict)

In [ ]:
stop

In [ ]:
vali_dict = camels.load_CV_snapshots(
    "CV_1",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
    return_hydro=True,
)

In [ ]:
run_simulations(vali_dict)

### test

In [ ]:
def plot_loss_of_scale(gas_poss, title="", out_dir=None):
    delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
    pos_loss = jnp.sum(delta_pos**2, axis=-1)
    pos_loss = jnp.mean(pos_loss, axis=1)
        
    res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
    kbins, res_cls = vpower_spectrum(res_rho)
    
    k = kbins[0]
    k_min, k_cutoff = k[0], k[int(0.2*len(k))]
    k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)
    cl_loss = jnp.sum(((res_cls/ref_cls - 1)*k_weights)**2, axis=-1)

    fig, ax = plt.subplots()

    ax.plot(scales, pos_loss, label="pos")
    ax.plot(scales, 0.1*cl_loss, label="cl")

    ax.legend()
    ax.set(xscale="linear", yscale="linear", xlabel="a", ylabel="loss", title=title)

    if out_dir is not None:
        plt.savefig(out_dir + ".png", dpi=100, bbox_inches="tight")


In [ ]:
with jax.default_device(jax.devices("cpu")[0]):
    plot_loss_of_scale(og_gas_poss, "gravity", f"plots/loss_gravity_i={i_snapshots},cnn")

In [ ]:
with jax.default_device(jax.devices("cpu")[0]):
    plot_loss_of_scale(nn_gas_poss, "gravity + pressure", f"plots/loss_gravity+pressure_i={i_snapshots},cnn")

In [ ]:
particle_loss_fn(model, architecture)

In [ ]:
particle_loss_fn(untrained_model, architecture)

In [ ]:
1/scales**2